# Proposed validation — review before it runs

**What this measures:** `l0_deficit_from_k` = k (64) minus the mean per-token L0 under the new `!=0` counting on a genuinely trained AbsTopK, whose encoder pre-activations are captured by a forward hook on the SAE's own `hook_sae_acts_pre` HookPoint — the one point both TopK and AbsTopK `encode()` route `sae_in @ W_enc + b_enc` through, since these SAEs hold W_enc as a raw Parameter and expose no `nn.Linear` in `sae.modules()` to find. The matched TopK control trains in the same run through the repo's own `LanguageModelSAETrainingRunner` into one engine-routed W&B group (log_to_wandb on LoggingConfig, per the reviewer's commit), and the guardrail `relu_topk_l0_predicate_delta` re-checks that the `>0` → `!=0` swap stays an exact no-op on that ReLU-family control (capability question, `baseline: none` — main cannot import AbsTopK, so no sentinel number is ever emitted).

**Target metric:** `l0_deficit_from_k`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

The executable copy lives at `benchmark/bench_abstopk_mechanism.py`, which is what `.remyx/validation.yaml` points at; keep the two in step, or point `suite:` here if you would rather maintain the notebook.

In [1]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

In [2]:
# Parameters
variant = "feature"
ref = "bf39b7abdf38400292f5edca911a206e111410fb"
seed = 0


### Weights & Biases

This validation logs training runs to **Weights & Biases** so the dashboards the repository asks for exist. It needs your W&B API key (W&B → Settings → API keys). In Colab, store it as a secret named `WANDB_API_KEY`; elsewhere set the environment variable. The run goes to entity `smellslikeml` (your W&B username or a team you belong to) and project `remyx-validate-saelens` (the folder the runs are grouped under, created if it does not exist) — the same place the Remyx run logs to, so your run appears beside it. Change the two values below to log somewhere else.

In [3]:
import os
if not os.environ.get("WANDB_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except Exception:
        pass
os.environ.setdefault("WANDB_ENTITY", "smellslikeml")
os.environ.setdefault("WANDB_PROJECT", "remyx-validate-saelens")
os.environ.setdefault("WANDB_BASE_URL", "https://api.wandb.ai")
os.environ.setdefault("WANDB_TAGS", "remyx-validate,hand-run")
if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_MODE"] = "disabled"
    print("WANDB_API_KEY not set — W&B logging disabled for this run")
else:
    print(f"W&B: {os.environ['WANDB_ENTITY']}/{os.environ['WANDB_PROJECT']}")

W&B: smellslikeml/remyx-validate-saelens


## Execution context

The cells below are the script at `benchmark/bench_abstopk_mechanism.py`, unchanged. This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [4]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "benchmark/bench_abstopk_mechanism.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

[remyx] cwd /workspace/target_repo | script /workspace/target_repo/benchmark/bench_abstopk_mechanism.py


In [5]:
"""Benchmark: does the AbsTopK port keep exactly the k largest-magnitude
signed encoder pre-activations per token, and is the >0 -> !=0 eval-counting
swap an exact no-op for the ReLU-family (TopK) control?

Both arms train through the repo's own LanguageModelSAETrainingRunner with one
shared config (gpt2 / blocks.6.hook_resid_post / streaming pile / 4096-token
batches / cpu); only sae= differs (topk control vs abstopk test), and the
runner's own log_to_wandb (LoggingConfig, per the reviewer's wiring) creates
one W&B run per arm from the engine's env — this script never touches WANDB_*.

Pre-activation capture (user guidance): a forward hook on the SAE's own
hook_sae_acts_pre HookPoint, the one point both TopK and AbsTopK encode() route
`sae_in @ W_enc + b_enc` through — SAELens SAEs keep W_enc as a raw Parameter,
so no nn.Linear exists in sae.modules() to find.

REMYX_SMOKE=1 shrinks to 2 optimizer steps and 64 eval tokens; same JSON keys.
"""

"Benchmark: does the AbsTopK port keep exactly the k largest-magnitude\nsigned encoder pre-activations per token, and is the >0 -> !=0 eval-counting\nswap an exact no-op for the ReLU-family (TopK) control?\n\nBoth arms train through the repo's own LanguageModelSAETrainingRunner with one\nshared config (gpt2 / blocks.6.hook_resid_post / streaming pile / 4096-token\nbatches / cpu); only sae= differs (topk control vs abstopk test), and the\nrunner's own log_to_wandb (LoggingConfig, per the reviewer's wiring) creates\none W&B run per arm from the engine's env — this script never touches WANDB_*.\n\nPre-activation capture (user guidance): a forward hook on the SAE's own\nhook_sae_acts_pre HookPoint, the one point both TopK and AbsTopK encode() route\n`sae_in @ W_enc + b_enc` through — SAELens SAEs keep W_enc as a raw Parameter,\nso no nn.Linear exists in sae.modules() to find.\n\nREMYX_SMOKE=1 shrinks to 2 optimizer steps and 64 eval tokens; same JSON keys.\n"

In [6]:
import argparse
import json
import os
import sys
from pathlib import Path

import torch

In [7]:
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from sae_lens import (
    LanguageModelSAERunnerConfig,
    LanguageModelSAETrainingRunner,
    LoggingConfig,
    TopKTrainingSAEConfig,
)

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Exactly the symbol this PR adds; absent on a checkout without the change.
try:
    from sae_lens.saes.abstopk_sae import AbsTopKTrainingSAEConfig

    ABSTOPK_AVAILABLE = True
except (ImportError, AttributeError):
    AbsTopKTrainingSAEConfig = None
    ABSTOPK_AVAILABLE = False

In [9]:
HOOK = "blocks.6.hook_resid_post"
K, D_IN, D_SAE = 64, 768, 8192  # PR pre-registered sweep point (VALIDATION.md)
SMOKE = os.environ.get("REMYX_SMOKE") == "1"
TRAIN_TOKENS = 2 * 4096 if SMOKE else 500_000  # ~122 steps/arm at 4096 tok/step
N_EVAL_TOKENS = 64 if SMOKE else 512  # 512 tokens x k=64 = 32k actives for the 0.5 share
EVAL_TEXT = "The quick brown fox jumps over the lazy dog. " * 64  # fixed held-out text

In [10]:
def _wandb_run_dir_count() -> int:
    # Filesystem evidence that the runner's wandb.init fired (wandb writes
    # ./wandb/run-*/ per initialized run); no dashboard and no env reads.
    return len(list(Path("wandb").glob("run-*"))) if Path("wandb").is_dir() else 0

def _train(sae_cfg, seed):
    torch.manual_seed(seed)
    cfg = LanguageModelSAERunnerConfig(
        sae=sae_cfg,
        model_name="gpt2",
        hook_name=HOOK,
        dataset_path="monology/pile-uncopyrighted",
        streaming=True,
        context_size=256,
        train_batch_size_tokens=4096,
        training_tokens=TRAIN_TOKENS,
        device="cpu",
        logger=LoggingConfig(log_to_wandb=True),  # the runner config's field is `logger`
    )
    return LanguageModelSAETrainingRunner(cfg).run()

In [11]:
@torch.no_grad()
def _measure(sae, x: torch.Tensor) -> dict:
    sae.eval()
    captured: list[torch.Tensor] = []

    def _hook(value, hook=None):
        # HookPoint.add_hook registers a TransformerLens activation hook,
        # invoked as hook(acts, hook=<HookPoint>); its return value REPLACES
        # the acts in HookPoint.forward, so pass the tensor through unchanged.
        captured.append(value.detach().to(torch.float32))
        return value

    sae.hook_sae_acts_pre.add_hook(_hook)
    acts = sae.encode(x).detach().to(torch.float32)
    sae.hook_sae_acts_pre.remove_hooks()
    pre = captured[0]

    def _m(t: torch.Tensor) -> float:
        return t.float().mean().item() if t.numel() else 0.0

    kept = acts != 0  # the new eval-counting predicate
    new_l0 = kept.sum(-1).float().mean().item()
    old_l0 = (acts > 0).sum(-1).float().mean().item()  # the old predicate
    top_idx = pre.abs().topk(K, dim=-1).indices  # k largest-magnitude pre-acts
    top_mask = torch.zeros_like(pre, dtype=torch.long).scatter_(-1, top_idx, 1)
    recon = sae.decode(sae.encode(x))
    return {
        "l0_deficit_from_k": K - new_l0,  # 0.0 = exactly k signed actives survived
        "predicate_delta": abs(new_l0 - old_l0),  # 0.0 = >0 vs !=0 no-op
        "bidirectionality_deviation": abs(_m(acts[kept] < 0) - 0.5),
        "undercount_ratio": old_l0 / new_l0 if new_l0 > 0 else 1.0,
        "magnitude_selection_exact": (kept.to(torch.long) == top_mask)
        .all(-1)
        .float()
        .mean()
        .item(),
        "sign_preservation_exact": _m(torch.sign(acts[kept]) == torch.sign(pre[kept])),
        "mse": ((recon - x) ** 2).mean().item(),
    }

In [12]:
def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--variant", default="")  # informational only
    parser.add_argument("--ref", default="")  # informational only
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    os.makedirs("checkpoints", exist_ok=True)  # runner's checkpoint destination

    runs_before = _wandb_run_dir_count()
    control = _train(TopKTrainingSAEConfig(k=K, d_in=D_IN, d_sae=D_SAE), args.seed)
    control_logged = _wandb_run_dir_count() > runs_before

    test = None
    test_logged = False
    if ABSTOPK_AVAILABLE:  # runner resolves architecture "abstopk" via the registry
        mid = _wandb_run_dir_count()
        test = _train(AbsTopKTrainingSAEConfig(k=K, d_in=D_IN, d_sae=D_SAE), args.seed)
        test_logged = _wandb_run_dir_count() > mid

    from transformer_lens import HookedTransformer

    model = HookedTransformer.from_pretrained("gpt2")
    tokens = model.to_tokens(EVAL_TEXT)[0]
    _, cache = model.run_with_cache(tokens.unsqueeze(0))
    x = cache[HOOK].reshape(-1, D_IN).to(torch.float32)[:N_EVAL_TOKENS]
    del model, cache  # identical batches fed to both arms below

    ctl = _measure(control, x)
    tst = _measure(test, x) if test is not None else None

    metrics = {
        "l0_deficit_from_k": tst["l0_deficit_from_k"] if tst else float(K),
        "relu_topk_l0_predicate_delta": ctl["predicate_delta"],
        "bidirectionality_deviation": tst["bidirectionality_deviation"] if tst else 0.5,
        "old_predicate_undercount_ratio": tst["undercount_ratio"] if tst else 1.0,
        "magnitude_selection_exact": tst["magnitude_selection_exact"] if tst else 0.0,
        "sign_preservation_exact": tst["sign_preservation_exact"] if tst else 0.0,
        "mse_ratio_abstopk_vs_topk": (
            tst["mse"] / ctl["mse"] if tst and ctl["mse"] > 0 else 999.0
        ),
        "abstopk_registered": 1.0 if tst else 0.0,
        "wandb_runs_logged": float(control_logged) + float(test_logged),
    }
    print(json.dumps(metrics))

if __name__ == "__main__":
    main()

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'monology/pile-uncopyrighted' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loaded pretrained model gpt2 into HookedTransformer


/workspace/target_repo/sae_lens/training/activations_store.py:465: UserWarning: Dataset is not tokenized. Pre-tokenizing will improve performance and allows for more control over special tokens. See https://decoderesearch.github.io/SAELens/training_saes/#pretokenizing-datasets for more info.
  warnings.warn(
[remyx] wandb: project 'sae_lens_training' -> 'remyx-validate-saelens' (the yaml's destination wins)


[remyx] wandb: init -> smellslikeml/remyx-validate-saelens id=db7a163b-feature-s0 group=db7a163b-3ad6-4a21-9934-ba99a6b957a5


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


...

wandb: 🚀 View run abstopk-8192-LR-0.0003-Tokens-5.000e+05 at: https://wandb.ai/smellslikeml/remyx-validate-saelens/runs/db7a163b-feature-s0-2
wandb: ⭐️ View project at: https://wandb.ai/smellslikeml/remyx-validate-saelens
wandb: Synced 4 W&B file(s), 0 media file(s), 5 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260920_000636-db7a163b-feature-s0-2/logs


/tmp/ipykernel_14/1675606618.py:22: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("gpt2")


Loaded pretrained model gpt2 into HookedTransformer


{"l0_deficit_from_k": 0.0, "relu_topk_l0_predicate_delta": 0.0, "bidirectionality_deviation": 0.0433349609375, "old_predicate_undercount_ratio": 0.5433349609375, "magnitude_selection_exact": 0.0, "sign_preservation_exact": 1.0, "mse_ratio_abstopk_vs_topk": 1.0015392062699298, "abstopk_registered": 1.0, "wandb_runs_logged": 2.0}


## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
question:
  kind: capability
  ask: "@remyx-ai validate — does the AbsTopK port select exactly the k largest-magnitude pre-activations per token with signs preserved (~half negative) on real gpt2 activations, and is the >0 -> !=0 eval-counting fix an exact no-op for ReLU variants?"
loop: {max_iterations: 8, fix_code: true}
requires: [wandb]
benchmarks:
  - name: abstopk-mechanism
    suite: "benchmark/bench_abstopk_mechanism.py"
    packages: [wandb]
    baseline: none
    scorer: l0_deficit_from_k
    policy: {guardrail_veto: true}
    # timeout derivation: 2 arms x ~122 steps (500k tokens / 4096 tokens-per-step)
    # x ~8-10 s/step on CPU ~= 35-40 min training + gpt2 download + pile-stream
    # warmup + held-out eval -> 5400 s ceiling for one job
    compute: {tier: cpu, timeout_s: 5400}
    metrics:
      - name: l0_deficit_from_k
        direction: min
        threshold: 0.0
        role: target
        bar: goal
        reads_as: "k minus mean per-token L0 under the new !=0 counting on the TRAINED AbsTopK; 0.0 = exactly k signed features survived (sign-killing reads ~k/2)"
      - name: relu_topk_l0_predicate_delta
        direction: min
        threshold: 0.5
        role: guardrail
        bar: goal
        reads_as: "absolute per-token L0 shift on the ReLU-family control from swapping >0 for !=0; correct arithmetic gives exactly 0.0, and the 0.5 bound (half an active feature per token) is a real gate — any wholesale predicate regression (>= 0 counting padding zeros, < 0 counting nothing) shifts L0 by >= 1.0, up to the full k=64, and fails it; measured on the trained TopK control, which exists identically on the baseline checkout, so the guardrail is baseline-achievable"
      - name: bidirectionality_deviation
        direction: min
        threshold: 0.15
        role: diagnostic
        reads_as: "|negative share of TRAINED active features - 0.5|; random-init 3-sigma band was 0.0084, trained features may specialize — 0.15 is a reading bar, not a gate"
      - name: old_predicate_undercount_ratio
        direction: min
        threshold: 0.9
        role: diagnostic
        reads_as: "old >0-counted L0 over new !=0-counted L0 on trained AbsTopK acts; must be <= 0.9 to demonstrate a real undercount — expected ~0.5 (ratio = 1 - negative share); 1.0 would mean no negative actives at all (sign-killing), i.e. the undercount the fix corrects does not exist"
      - name: magnitude_selection_exact
        direction: max
        threshold: 1.0
        role: diagnostic
        reads_as: "share of held-out tokens whose kept set equals exactly the k largest-magnitude pre-activations"
      - name: sign_preservation_exact
        direction: max
        threshold: 1.0
        role: diagnostic
        reads_as: "share of active features whose output sign matches their pre-activation sign"
      - name: mse_ratio_abstopk_vs_topk
        direction: min
        threshold: 1.0
        role: diagnostic
        bar: goal
        reads_as: "trained-AbsTopK MSE over trained-TopK MSE at matched k=64 on held-out gpt2 activations; paper claims <= 1 at scale — directional at 500k tokens"
      - name: abstopk_registered
        direction: max
        threshold: 1.0
        role: diagnostic
        bar: sanity
        reads_as: "1.0 when the repo runner resolves architecture abstopk end-to-end (the PR's registration block + AbsTopKTrainingSAE construction during a completed training arm); 0.0 on a checkout where the symbols are absent"
      - name: wandb_runs_logged
        direction: max
        threshold: 2.0
        role: diagnostic
        bar: sanity
        reads_as: "arms whose wandb run initialized and finished inside the engine's run group; 2.0 = control + test dashboards exist to link from the PR"
    held_constant:
      - "one LanguageModelSAERunnerConfig for both arms — model gpt2, hook blocks.6.hook_resid_post, dataset monology/pile-uncopyrighted (streaming), context_size 256, train_batch_size_tokens 4096, training_tokens 500k (a real optimizer-driven run at the cpu tier; the VALIDATION.md protocol scales the same config to 500M-1B tokens on GPU), device cpu — only the sae= architecture differs (topk control vs abstopk test)"
      - "k=64, d_sae=8192, d_in=768 — the PR pre-registered sweep point (VALIDATION.md)"
      - "held-out eval on a fixed text through the trained SAEs; identical batches fed to the AbsTopK test and the TopK control; encoder pre-activations captured by a forward hook on the SAE's own hook_sae_acts_pre HookPoint fired during encode (add_hook -> append output -> remove) — the one capture point both encode() paths share, since sae_in @ W_enc + b_enc always routes through it; mechanism metrics computed on float32 tensors from the trained weights"
      - "both arms log to one W&B group: the engine sets WANDB_API_KEY/ENTITY/PROJECT/RUN_GROUP and the per-arm run identity; the script never reads or sets WANDB_* itself — the repo runner's own log_to_wandb (LoggingConfig, as the reviewer's notebook commit wires it) creates one run per arm"
    avoid:
      - "no sae.modules() scan for an nn.Linear encoder — SAELens SAEs keep W_enc/W_dec as raw Parameters and expose none; the pre-activation capture is a forward hook on the SAE's own hook_sae_acts_pre HookPoint, which both TopK and AbsTopK encode() route through (user guidance)"
      - "the scorer never reads the wandb dashboard — metrics come from the script's JSON line; the dashboard is linked evidence for the PR template, not the instrument"
      - "run_evals is not driven directly (ActivationScaler/ActivationsStore constructor signatures are outside the visible context); training itself goes through the repo runner, and the changed !=0/>0 predicates are measured on the real trained-operator outputs"
      - "no hardcoded wandb entity or project, no WANDB_* reads, no id/group/mode set and no second wandb.init in the script — routing and per-arm run identity come from the engine's env and the runner's own logging"
      - "no wall-clock timing and no cuda — tier: cpu provisions no GPU; the fixed seed keeps the run reproducible up to streaming-buffer order, and the exact-k/sign/predicate metrics are drift-free"
      - "no unpinned live APIs: gpt2 weights and the pile stream are fetched by the repo's own loaders (model-name/dataset-path pinned, HF-cached); no CE/KL evals, no interpretability probing"
    report:
      headline: "A real W&B-logged training run (one group: TopK control vs AbsTopK test) trains both arms end-to-end through the repo runner on CPU; the mechanism holds on the trained AbsTopK — exactly-k magnitude selection, signs preserved, !=0 L0 counting negative actives — with short-run MSE parity directional at 500k tokens"
      findings:
        - "per-token L0 under the new !=0 counting falls {l0_deficit_from_k} short of k=64 on the trained AbsTopK (0.0 expected; ~32 would indicate sign-killing)"
        - "the >0 -> !=0 swap changes the trained TopK control's L0 by {relu_topk_l0_predicate_delta} (exactly 0.0 expected; guardrail bound 0.5, so any wholesale predicate regression reading >= 1.0 fails)"
        - "the negative share of trained active features sits {bidirectionality_deviation} from 0.5 (random-init 3-sigma band was 0.0084; trained features may specialize)"
        - "the old >0 counting reports {old_predicate_undercount_ratio} of the new L0 on trained AbsTopK acts (~0.5 = the corrected 2x undercount; bound 0.9, 1.0 = no negative actives)"
        - "{magnitude_selection_exact} of held-out tokens keep exactly the k largest-magnitude pre-activations"
        - "{sign_preservation_exact} of active features retain their pre-activation sign"
        - "trained AbsTopK reconstruction MSE is {mse_ratio_abstopk_vs_topk} times the trained TopK control's at matched k=64 (paper claims <= 1 at scale; 500k tokens is directional)"
        - "{abstopk_registered} registry wiring for architecture abstopk; {wandb_runs_logged} wandb runs (2.0 = control + test) landed in the engine's run group"
      establishes:
        - "AbsTopK trains end-to-end through the repo's own LanguageModelSAETrainingRunner and logs beside a matched TopK control into one W&B group — the control/test dashboards the PR template requires"
        - "on trained weights at the PR's pre-registered geometry, the operator still selects exactly the k largest-magnitude pre-activations with signs preserved, and the !=0 eval counting reports them"
        - "the >0 -> !=0 predicate swap remains an exact no-op for the trained ReLU-family (TopK) control"
      does_not_establish:
        - "full-scale sparsity-fidelity parity (500M-1B tokens on GPU, k in {16,64,256}, JumpReLU control) — the run the PR itself defers; the 500k-token MSE ratio is directional only"
        - "any interpretability claim: probing, steering, or single features encoding contrasting concepts"
        - "behaviour of the full run_evals loop through ActivationsStore — only its changed predicates are exercised on trained outputs"
        - "the latent issue that evals.py l1 is a signed sum and cancels toward 0 for AbsTopK (untouched by this PR)"
      not_measured:
        - "ce_loss_score, kl_div, explained_variance, dead-feature counts beyond the trainer's own wandb curves, throughput/wall-clock, feature-dashboard interpretability"
      caveat: "the paper's headline results (7 probing/steering tasks across 4 LLMs, matching supervised DiM) have no harness in this repo; this run supplies the template's control/test dashboards plus mechanism evidence, not the deferred parity frontier"
      next: "run the full VALIDATION.md protocol on GPU: shared activation cache, k in {16,64,256}, TopK and JumpReLU controls, 500M-1B training tokens, run_evals l0/mse/ce_loss_score plus dead-feature counts at matched L0"
    provenance:
      l0_deficit_from_k: "claim_analysis"
      relu_topk_l0_predicate_delta: "claim_analysis"
      bidirectionality_deviation: "claim_analysis"
      old_predicate_undercount_ratio: "claim_analysis"
      magnitude_selection_exact: "claim_analysis"
      sign_preservation_exact: "claim_analysis"
      mse_ratio_abstopk_vs_topk: "claim_analysis"
      abstopk_registered: "claim_analysis"
      wandb_runs_logged: "maintainer_comment:@salma-remyx"
      pre_activation_capture: "user_guidance"
      held_constant: "protocol_doc:VALIDATION.md"
      suite: "synthesized"
      compute: "inferred"
      requires: "maintainer_comment:@salma-remyx"
logging:
  wandb:
    entity: smellslikeml    # from your W&B connection
    project: remyx-validate-saelens    # default
```